# In-Memory RAG with `dspy.retrievers.Embeddings` (Chapter 8)

This notebook accompanies Chapter 8 §8.3.1 of *Context Engineering with DSPy*. You will embed a small corpus with `dspy.Embedder`, build a FAISS-backed semantic search index with `dspy.retrievers.Embeddings`, and wrap retrieval plus generation in a `SimpleRAG` module.

**Required environment variable**

- `OPENAI_API_KEY` — used for both the LM (`openai/gpt-5-mini`) and the embedder (`openai/text-embedding-3-small`).

Everything runs in-process; no extra services required.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5-mini"))

## Building the in-memory index

`dspy.retrievers.Embeddings` does brute-force cosine similarity for corpora under 20,000 documents and switches to a FAISS IVF-PQ index automatically beyond that. For prototyping a handful of strings is enough.

In [ ]:
# Your knowledge base or any list of strings
corpus = [
    "DSPy is a framework for programming language models.",
    "Signatures define input/output specifications for DSPy tasks.",
    "Optimizers like BootstrapFewShot automatically improve prompts.",
    "ReAct agents use tools to gather information iteratively.",
    "GEPA is a reflective evolutionary optimizer for DSPy programs.",
    "dspy.History stores conversation turns as a list of message dicts.",
]

# Create a semantic search index
embedder = dspy.Embedder("openai/text-embedding-3-small", dimensions=512)
search = dspy.retrievers.Embeddings(
    embedder=embedder,
    corpus=corpus,
    k=3,  # return top 3 results
)

# Search the index
results = search("What are DSPy optimizers?")
print(results.passages)

## Wrapping retrieval and generation in a module

A RAG pipeline in DSPy is a regular `dspy.Module` that takes a retriever in its constructor and pairs the retrieved passages with a `ChainOfThought` step. Because the retriever is swappable, the same module works for any backend later in the chapter.

In [ ]:
class SimpleRAG(dspy.Module):
    def __init__(self, retriever):
        super().__init__()
        self.retriever = retriever
        self.respond = dspy.ChainOfThought("context, question -> response")

    def forward(self, question):
        context = self.retriever(question).passages
        return self.respond(context=context, question=question)


rag = SimpleRAG(retriever=search)
result = rag(question="How do I optimize a DSPy program?")
print(result.response)